This file is part of GaPSE
Copyright (C) 2022 Matteo Foglieni

GaPSE is free software: you can redistribute it and/or modify
it under the terms of the GNU General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

GaPSE is distributed in the hope that it will be useful, but
WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU
General Public License for more details.

You should have received a copy of the GNU General Public License
along with GaPSE. If not, see <http://www.gnu.org/licenses/>.

# Iln_terms

Here we plot all the $I_\ell^n$ integrals that GaPSE uses to build every Two-Point Correlation Function (TPCF), together with their asymptotic behaviour for $s \rightarrow 0$.

All the plots and the data are saved in the `Iln_terms/` directory (check to have it, otherwise create it). There is also the script `Iln_terms.jl` that performs the same computations from the terminal.

The derivation of the limits shown here is *not* repeated: you can find it in the "The $I_\ell^n$ integrals" and "The $\Delta\chi \rightarrow 0$ limits" pages of the GaPSE documentation.

## The theory in a nutshell

Every $\xi$ that GaPSE computes is a sum of terms of the form $J(\chi, s, y) \, I_\ell^n(\Delta\chi)$, where

$$
\begin{equation}
    I_\ell^n(s) := \int_0^{+\infty} \frac{\mathrm{d}q}{2\pi^2} \, q^2 \, P(q) \, \frac{j_\ell(qs)}{(qs)^n} \; ,
\end{equation}
$$

with $P(q)$ the matter Power Spectrum at $z=0$ and $j_\ell$ the spherical Bessel function of order $\ell$. The eight combinations $(\ell, n)$ that appear in the code are

$$
(0,0) \, , \; (2,0) \, , \; (4,0) \, , \; (0,2) \, , \; (2,2) \, , \; (3,1) \, , \; (1,3) \, , \; (1,1) \; ,
$$

together with the auxiliary

$$
\begin{align}
    \tilde{I}_0^4(s) &:= \int_0^{+\infty} \frac{\mathrm{d}q}{2\pi^2} \, q^2 \, P(q) \, \frac{j_0(qs) - 1 }{(qs)^4} \\[10pt]
        &= \frac{1}{s^4} \int_0^{+\infty} \frac{\mathrm{d}q}{2\pi^2} \, \frac{P(q)}{q^2} \, \left[ j_0(qs) - 1 \right] \; .
\end{align}
$$

It is convenient to introduce the moments of the Power Spectrum

$$
\begin{equation}
    \sigma_i = \int_{k_\mathrm{min}}^{k_\mathrm{max}} \frac{\mathrm{d}q}{2 \pi^2} \, q^{2-i} \, P(q) \; ,
\end{equation}
$$

which are the same $\sigma_i$ stored inside `IPSTools`. With them, the behaviour of the $I_\ell^n$ for small separations is

$$
\begin{equation}
    I_\ell^n(s) \; \xrightarrow[s \rightarrow 0]{} \;
        \frac{\sigma_{n-\ell}}{(2\ell+1)!!} \, s^{\,\ell - n} \; ,
    \qquad
    \tilde{I}_0^4(s) \; \xrightarrow[s \rightarrow 0]{} \; - \frac{\sigma_2}{6 \, s^2} \; ,
\end{equation}
$$

so that there are three possible regimes:

- $\ell > n$ : the integral **vanishes** as $s^{\ell-n}$;
- $\ell = n$ : the integral tends to the **finite** value $\sigma_0/(2\ell+1)!!$;
- $\ell < n$ : the integral **diverges** as $s^{-(n-\ell)}$.

Explicitly:

| | $\ell$ | $n$ | $s \rightarrow 0$ |
|:--|:-:|:-:|:--|
| $I_0^0$ | 0 | 0 | $\sigma_0$ |
| $I_2^0$ | 2 | 0 | $\sigma_{-2} \, s^2 / 15$ |
| $I_4^0$ | 4 | 0 | $\sigma_{-4} \, s^4 / 945$ |
| $I_0^2$ | 0 | 2 | $\sigma_2 / s^2$ |
| $I_2^2$ | 2 | 2 | $\sigma_0 / 15$ |
| $I_3^1$ | 3 | 1 | $\sigma_{-2} \, s^2 / 105$ |
| $I_1^3$ | 1 | 3 | $\sigma_2 / (3 s^2)$ |
| $I_1^1$ | 1 | 1 | $\sigma_0 / 3$ |
| $\tilde{I}_0^4$ | - | - | $-\sigma_2 / (6 s^2)$ |

The divergent ones are never a problem in practice, because in the TPCFs they always come multiplied by a $J$ carrying the matching positive power of $\Delta\chi$.

In [ ]:
using Pkg
Pkg.activate(@__DIR__)
using GaPSE

using Plots, LaTeXStrings, QuadGK, DelimitedFiles, Printf

pyplot() # if you do not have PyPlot/matplotlib installed, `gr()` works as well

In [ ]:
const PATH_TO_GAPSE = normpath(joinpath(@__DIR__, ".."))

# Input matter Power Spectrum P(q) at z=0
const FILE_PS = joinpath(PATH_TO_GAPSE, "data", "WideA_ZA_pk.dat")

# Directory where the plots and the data will be saved
const DIR = joinpath(@__DIR__, "Iln_terms")
@assert isdir(DIR) "ERROR: DIR=$DIR DOESN'T EXIST!!!"

# Set this to `true` in order to save a copy of the plots where the
# documentation expects to find them.
const SAVE_TO_DOCS = true
const DOCS_ASSETS = joinpath(PATH_TO_GAPSE, "docs", "src", "assets", "Iln_terms")
SAVE_TO_DOCS && mkpath(DOCS_ASSETS)

# Integration extremes of the sigma_i, the same defaults used by `IPSTools`
const K_MIN, K_MAX = 1e-6, 10.0

# Comoving separations where the I_l^n will be evaluated
const SS = 10 .^ range(-4, 4, length=600);

We build the input Power Spectrum and the `IPSTools` that contains all the $I_\ell^n$. Note that they are `IntegralIPS` objects: they are splines in their sampled range, and power laws outside of it, so they can be safely evaluated at any $s>0$.

In [ ]:
ips = GaPSE.InputPS(FILE_PS)
tools = GaPSE.IPSTools(ips; k_min=K_MIN, k_max=K_MAX, N=1024,
    fit_min=0.05, fit_max=0.5, con=true);

`IPSTools` stores only $\sigma_0, ..., \sigma_4$, while the asymptotes of $I_2^0$, $I_4^0$ and $I_3^1$ need the negative-index ones, so we recompute them here.

In [ ]:
sigma(i) = quadgk(q -> ips(q) * q^(2 - i) / (2 * π^2), K_MIN, K_MAX)[1]

dfact(n) = n <= 0 ? 1 : prod(n:-2:1)   # the double factorial n!!

# name, l, n, the IntegralIPS stored in `tools`
const ILN = [
    ("I00", 0, 0, tools.I00),
    ("I20", 2, 0, tools.I20),
    ("I40", 4, 0, tools.I40),
    ("I02", 0, 2, tools.I02),
    ("I22", 2, 2, tools.I22),
    ("I31", 3, 1, tools.I31),
    ("I13", 1, 3, tools.I13),
    ("I11", 1, 1, tools.I11),
]

asymptote(s, l, n) = sigma(n - l) * s^(l - n) / dfact(2 * l + 1)
asymptote_tilde(s) = -sigma(2) / (6 * s^2);

In [ ]:
@printf("sigma_-4 = %.6e \n", sigma(-4))
@printf("sigma_-2 = %.6e \n", sigma(-2))
@printf("sigma_0  = %.6e \t (IPSTools: %.6e) \n", sigma(0), tools.σ_0)
@printf("sigma_2  = %.6e \t (IPSTools: %.6e) \n", sigma(2), tools.σ_2)

## The plots

We plot $|I_\ell^n(s)|$ rather than $I_\ell^n(s)$: all these integrals oscillate and change sign at large $s$, where a logarithmic vertical axis would not be defined. At small $s$, which is the regime we care about here, they do not change sign, so the absolute value is immaterial and the asymptote (dashed, black) can be compared directly.

In [ ]:
const LOGTICKS = (
    vcat([a * 10.0^b for b in -4:3 for a in 1:9], 10.0^4),
    vcat([a == 1 ? L"10^{%$b}" : nothing for b in -4:3 for a in 1:9], L"10^{4}")
)

plot_kwargs() = Dict(
    :xaxis => :log, :yaxis => :log,
    :xlabel => L"s \quad [h_0^{-1}\mathrm{Mpc}]",
    :xticks => LOGTICKS,
    :legend => :bottomleft,
    :size => (600, 400),
)

function plot_single(name, l, n, f; tilde=false)
    ys = [f(s) for s in SS]
    lab = tilde ? L"|\tilde{I}_0^4(s)|" : L"|I_{%$l}^{%$n}(s)|"
    asy = tilde ? [asymptote_tilde(s) for s in SS] : [asymptote(s, l, n) for s in SS]
    asylab = tilde ? L"|-\sigma_2 / (6 s^2)|" :
             L"|\sigma_{%$(n-l)} \, s^{%$(l-n)} / (2 \cdot %$l + 1)!!|"

    p = plot(SS, abs.(ys); label=lab, lw=2,
        ylabel=tilde ? L"|\tilde{I}_0^4(s)|" : L"|I_{\ell}^{n}(s)|",
        title=tilde ? L"\tilde{I}_0^4" : L"I_{%$l}^{%$n}", plot_kwargs()...)
    plot!(p, SS, abs.(asy); label=asylab, ls=:dash, lw=2, color=:black)

    savefig(p, joinpath(DIR, name * ".png"))
    SAVE_TO_DOCS && savefig(p, joinpath(DOCS_ASSETS, name * ".png"))
    return p
end;

In [ ]:
ps = [plot_single(name, l, n, f) for (name, l, n, f) in ILN]
push!(ps, plot_single("I04_tilde", 0, 4, tools.I04_tilde; tilde=true))
plot(ps..., layout=(3, 3), size=(1500, 1100))

And now all of them in a single figure. The three regimes are immediately visible: the curves that flatten out ($\ell = n$), the ones that go down as a power law ($\ell > n$) and the ones that blow up ($\ell < n$).

In [ ]:
function plot_all()
    p = plot(; ylabel=L"|I_{\ell}^{n}(s)|", title=L"\mathrm{All \; the} \; I_{\ell}^{n}",
        plot_kwargs()...)
    for (name, l, n, f) in ILN
        plot!(p, SS, abs.([f(s) for s in SS]); label=L"I_{%$l}^{%$n}", lw=2)
    end
    plot!(p, SS, abs.([tools.I04_tilde(s) for s in SS]);
        label=L"\tilde{I}_0^4", lw=2, ls=:dot)

    savefig(p, joinpath(DIR, "all_Iln.png"))
    SAVE_TO_DOCS && savefig(p, joinpath(DOCS_ASSETS, "all_Iln.png"))
    return p
end

plot_all()

Finally we save the numerical values, so that the figures can be re-made without recomputing the `IPSTools`.

In [ ]:
out = joinpath(DIR, "Iln_values.txt")
isfile(out) && rm(out)
open(out, "w") do io
    println(io, GaPSE.BRAND)
    println(io, "#\n# The I_l^n integrals evaluated in the following comoving separations.")
    println(io, "# Input Power Spectrum file: $FILE_PS")
    println(io, "# k_min = $K_MIN , k_max = $K_MAX")
    println(io, "#\n# sigma_0 = $(tools.σ_0) \t sigma_2 = $(tools.σ_2)")
    println(io, "# sigma_-2 = $(sigma(-2)) \t sigma_-4 = $(sigma(-4))")
    println(io, "#")
    println(io, "# s [h_0^{-1} Mpc] \t " * join([n for (n, _, _, _) in ILN], " \t ") * " \t I04_tilde")
    for s in SS
        vals = [f(s) for (_, _, _, f) in ILN]
        println(io, "$s \t " * join(vals, " \t ") * " \t $(tools.I04_tilde(s))")
    end
end
println("saved in $out")